<a href="https://colab.research.google.com/github/J-Aamir/FineTuned-Finance-Models-Evaluation-using-Judge/blob/main/Mistral_7b%20Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 1: Fine-tune Mistral-7B-Instruct-v0.3
# Financial QA | finance-alpaca dataset
# Run this in Google Colab (T4 GPU)
# ============================================================

# ── CELL 1: Check GPU ────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# ── CELL 2: Install dependencies ─────────────────────────────
# Run this cell first and restart runtime when prompted
get_ipython().system('pip install -q unsloth')
get_ipython().system('pip install -q datasets trl peft bitsandbytes transformers accelerate')
get_ipython().system('pip install -q wandb')  # optional, for logging

# ── CELL 3: Mount Google Drive (to save model) ───────────────
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = "/content/drive/MyDrive/assignment3/mistral7b_financial"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model will be saved to: {SAVE_DIR}")





Tue Apr 28 06:09:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── CELL 4: Load model with Unsloth (4-bit QLoRA) ────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024  # Reduced from 2048 to save VRAM
LOAD_IN_4BIT   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
print("✅ Mistral-7B loaded successfully")

# ── CELL 5: Attach LoRA adapters ─────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                = 16,          # LoRA rank
    target_modules   = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
    lora_alpha       = 32,
    lora_dropout     = 0.05,
    bias             = "none",
    use_gradient_checkpointing = "unsloth",  # saves memory
    random_state     = 42,
)
print("✅ LoRA adapters attached")
model.print_trainable_parameters()

# ── CELL 6: Load and prepare finance-alpaca dataset ──────────
from datasets import load_dataset

dataset = load_dataset("gbharti/finance-alpaca", split="train")
print(f"Total samples: {len(dataset)}")
print("Sample entry:")
print(dataset[0])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

✅ LoRA adapters attached
trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

Total samples: 68912
Sample entry:
{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36

In [ ]:
# ── CELL 7: Define prompt template ───────────────────────────
PROMPT_TEMPLATE = """Below is a financial question. Provide a clear, accurate, and helpful answer.

### Question:
{}

### Context:
{}

### Answer:
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_prompt(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        context = inp if inp and inp.strip() else "No additional context provided."
        text = PROMPT_TEMPLATE.format(inst, context, out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(format_prompt, batched=True)

# Filter out very short or very long samples
def filter_samples(example):
    token_count = len(tokenizer(example["text"])["input_ids"])
    return 200 < token_count < MAX_SEQ_LENGTH

dataset = dataset.filter(filter_samples)

# Use a subset for faster training on Colab T4
# Increase to 20000 if you get an A100 session
TRAIN_SIZE = 4000
dataset = dataset.shuffle(seed=42).select(range(min(TRAIN_SIZE, len(dataset))))
print(f"\n✅ Training on {len(dataset)} samples after filtering")

# Save 20 test questions separately (do NOT train on these)
# We'll use them in the evaluation notebook
TEST_QUESTIONS = [
    "What is the difference between a stock and a bond?",
    "How does compound interest work and why is it important?",
    "What is a P/E ratio and how is it used in stock valuation?",
    "Explain the concept of diversification in investing.",
    "What is inflation and how does it affect purchasing power?",
    "What is the difference between a bull market and a bear market?",
    "How do central banks use interest rates to control inflation?",
    "What is dollar-cost averaging and what are its benefits?",
    "Explain what an ETF is and how it differs from a mutual fund.",
    "What is the role of the Federal Reserve in the US economy?",
    "What is a balance sheet and what does it tell investors?",
    "How does quantitative easing work?",
    "What is the difference between gross profit and net profit?",
    "What is a credit score and what factors affect it?",
    "Explain the concept of short selling in the stock market.",
    "What is a hedge fund and how does it differ from a mutual fund?",
    "What is liquidity risk in financial markets?",
    "How do options contracts work in financial markets?",
    "What is the yield curve and what does an inverted yield curve signal?",
    "What is the difference between fiscal policy and monetary policy?"
]

import json
with open(f"{SAVE_DIR}/test_questions.json", "w") as f:
    json.dump(TEST_QUESTIONS, f, indent=2)
print(f"✅ Saved 20 test questions to Drive")


# ── CELL 8: Set up trainer (OPTIMIZED FOR T4 GPU & DRIVE) ─────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Checkpoints will go to: /content/drive/MyDrive/assignment3/mistral7b_financial/checkpoints
CHECKPOINT_DIR = os.path.join(SAVE_DIR, "checkpoints")

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LENGTH, # Ensure this is set to 1024 in Cell 4
    dataset_num_proc = 2,
    packing          = False,
    args = TrainingArguments(
        per_device_train_batch_size    = 1,      # Lowered to 1 to prevent OOM
        gradient_accumulation_steps    = 8,      # Effective batch size of 8
        warmup_steps                   = 50,
        max_steps                      = 300,    # Training for 300 steps (Fast & Effective)
        learning_rate                  = 2e-4,
        fp16                           = not is_bfloat16_supported(),
        bf16                           = is_bfloat16_supported(),
        logging_steps                  = 10,     # Frequent logging for your report graphs

        # --- DRIVE SAVING & RESUME ---
        output_dir                     = CHECKPOINT_DIR,
        save_strategy                  = "steps",
        save_steps                     = 100,            # Save to Drive every 100 steps
        save_total_limit               = 2,              # Only keep last 2 checkpoints to save Drive space

        optim                          = "adamw_8bit",   # Memory-efficient optimizer
        weight_decay                   = 0.01,
        lr_scheduler_type              = "cosine",
        seed                           = 42,
        report_to                      = "none",
    ),
)

print(f"✅ Trainer configured. Checkpoints will save to: {CHECKPOINT_DIR}")
print(f"✅ Memory protection active: Batch Size=1, Accumulation=8, Seq_Len={MAX_SEQ_LENGTH}")

Map:   0%|          | 0/68912 [00:00<?, ? examples/s]

Filter:   0%|          | 0/68912 [00:00<?, ? examples/s]


✅ Training on 4000 samples after filtering
✅ Saved 20 test questions to Drive


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4000 [00:00<?, ? examples/s]

✅ Trainer configured. Checkpoints will save to: /content/drive/MyDrive/assignment3/mistral7b_financial/checkpoints
✅ Memory protection active: Batch Size=1, Accumulation=8, Seq_Len=1024


In [ ]:
# ── CELL 9: Train (Smart Resume) ────────────────────
print("🚀 Starting training...")

import os
# Check if the checkpoint directory exists and has files in it
checkpoint_dir = trainer.args.output_dir
if os.path.exists(checkpoint_dir) and len(os.listdir(checkpoint_dir)) > 0:
    print("Found existing progress. Resuming...")
    resume = True
else:
    print("No checkpoints found. Starting from scratch...")
    resume = False

trainer_stats = trainer.train(resume_from_checkpoint=resume)

print("\n✅ Training complete!")
# ── CELL 10: Save model to Google Drive ───────────────────────
print("💾 Saving model to Google Drive...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Model saved to {SAVE_DIR}")

# Save training stats
import json
stats = {
    "model": "mistral-7b-instruct-v0.3",
    "dataset": "finance-alpaca",
    "train_samples": len(dataset),
    "final_loss": trainer_stats.training_loss,
    "epochs": 2,
}
with open(f"{SAVE_DIR}/training_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print("✅ Training stats saved")

🚀 Starting training...
No checkpoints found. Starting from scratch...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.153481
20,1.842073
30,1.725769
40,1.665075
50,1.714182
60,1.783809
70,1.707999


In [ ]:
# ── CELL 11: Quick inference test ─────────────────────────────
FastLanguageModel.for_inference(model)

def ask_model(question, context="No additional context provided."):
    prompt = PROMPT_TEMPLATE.format(question, context, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens    = 300,
        temperature       = 0.7,
        top_p             = 0.9,
        repetition_penalty = 1.1,
        do_sample         = True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part
    answer = response.split("### Answer:")[-1].strip()
    return answer

# Test with first question
q = TEST_QUESTIONS[0]
print(f"\nQuestion: {q}")
print(f"\nAnswer: {ask_model(q)}")

# ── CELL 12: Generate answers for all 20 test questions ───────
print("\n📝 Generating answers for all 20 test questions...")
mistral_answers = {}

for i, question in enumerate(TEST_QUESTIONS):
    print(f"  [{i+1}/20] {question[:60]}...")
    answer = ask_model(question)
    mistral_answers[question] = answer

# Save answers to Drive
with open(f"{SAVE_DIR}/mistral_answers.json", "w") as f:
    json.dump(mistral_answers, f, indent=2)

print(f"\n✅ All answers saved to {SAVE_DIR}/mistral_answers.json")
print("✅ NOTEBOOK 1 COMPLETE — proceed to Notebook 2 (Qwen2.5-7B)")